In [ ]:
import zipfile

zip_path = "/content/CV bonus project v1 5-4.v1-updated-data.yolov8.zip"
extract_path = "/content/bowling_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done")

Done


In [ ]:
import os
from pathlib import Path

DATASET_DIR = Path("/content/bowling_dataset")
YAML_PATH = DATASET_DIR / "data.yaml"

print("Dataset exists:", DATASET_DIR.exists())
print("data.yaml exists:", YAML_PATH.exists())

Dataset exists: True
data.yaml exists: True


In [ ]:
import yaml

with open(YAML_PATH, "r") as f:
    data_yaml = yaml.safe_load(f)

print(data_yaml)

names = data_yaml.get("names")
print("Classes:", names)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 4, 'names': ['ball', 'car', 'fallen-pins', 'standing-pins'], 'roboflow': {'workspace': 'ahmed-mohsen-on9f0', 'project': 'cv-bonus-project-v1-5-4', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/ahmed-mohsen-on9f0/cv-bonus-project-v1-5-4/dataset/1'}}
Classes: ['ball', 'car', 'fallen-pins', 'standing-pins']


In [ ]:
from collections import Counter

label_counter = Counter()

for split in ["train", "valid", "test"]:
    labels_dir = DATASET_DIR / split / "labels"

    if not labels_dir.exists():
        print(f"{split}: no labels folder")
        continue

    txt_files = list(labels_dir.glob("*.txt"))
    print(f"{split}: {len(txt_files)} label files")

    for txt in txt_files:
        with open(txt, "r") as f:
            for line in f:
                if line.strip():
                    cls_id = int(line.split()[0])
                    label_counter[cls_id] += 1

print("\nLabel count per class:")
for cls_id, count in label_counter.items():
    cls_name = names[cls_id] if isinstance(names, list) else names.get(cls_id, str(cls_id))
    print(cls_id, cls_name, count)

train: 1542 label files
valid: 220 label files
test: 111 label files

Label count per class:
3 standing-pins 12537
2 fallen-pins 6370
1 car 1378
0 ball 1752


In [ ]:
required = ["ball", "car", "standing-pins", "fallen-pins"]

all_names = list(names.values()) if isinstance(names, dict) else names

for cls in required:
    print(cls, "✅ found" if cls in all_names else "❌ missing")

ball ✅ found
car ✅ found
standing-pins ✅ found
fallen-pins ✅ found


In [ ]:
class_to_find = ["ball", "standing-pins", "fallen-pins"]

name_to_id = {v: k for k, v in names.items()} if isinstance(names, dict) else {v: i for i, v in enumerate(names)}

for target_class in class_to_find:
    if target_class not in name_to_id:
        print(f"\n{target_class}: class not found")
        continue

    target_id = int(name_to_id[target_class])
    found_files = []

    for split in ["train", "valid", "test"]:
        labels_dir = DATASET_DIR / split / "labels"
        if not labels_dir.exists():
            continue

        for txt in labels_dir.glob("*.txt"):
            with open(txt, "r") as f:
                if any(line.strip() and int(line.split()[0]) == target_id for line in f):
                    found_files.append(str(txt))

    print(f"\n{target_class}: found in {len(found_files)} label files")
    print(found_files[:10])


ball: found in 1031 label files
['/content/bowling_dataset/train/labels/VID_20260504_144957_idx00702_jpg.rf.439b1c3717c91032a7bef7cddd261d24.txt', '/content/bowling_dataset/train/labels/VID_20260504_135512_idx00426_jpg.rf.5197fd057374ff3b098e471f9abb285a.txt', '/content/bowling_dataset/train/labels/VID_20260504_143732_idx00321_jpg.rf.c3a34060752e071694bce6d93be5b2de.txt', '/content/bowling_dataset/train/labels/VID_20260504_143515_idx00273_jpg.rf.28985cfb3f972e4fac78107d4e4f9d88.txt', '/content/bowling_dataset/train/labels/VID_20260504_143515_idx00150_jpg.rf.8970a1adbfc373ba050de2a6588c43bd.txt', '/content/bowling_dataset/train/labels/VID_20260504_140731_idx00420_jpg.rf.7c4f35f083c6f2785d286ea40dfc1063.txt', '/content/bowling_dataset/train/labels/VID_20260504_144413_idx00072_jpg.rf.bb3be4f27efb520cd38cfbef456ef412.txt', '/content/bowling_dataset/train/labels/VID_20260504_135512_idx00237_jpg.rf.296f3d0ea7836c118415312f7bc082e5.txt', '/content/bowling_dataset/train/labels/VID_20260504_13

In [ ]:
import yaml

yaml_path = DATASET_DIR / "data.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

print(data)
print("Classes:", data["names"])

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 4, 'names': ['ball', 'car', 'fallen-pins', 'standing-pins'], 'roboflow': {'workspace': 'ahmed-mohsen-on9f0', 'project': 'cv-bonus-project-v1-5-4', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/ahmed-mohsen-on9f0/cv-bonus-project-v1-5-4/dataset/1'}}
Classes: ['ball', 'car', 'fallen-pins', 'standing-pins']


In [ ]:
fixed_yaml = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images" if (DATASET_DIR / "test/images").exists() else None,
    "names": data["names"]
}

if fixed_yaml["test"] is None:
    fixed_yaml.pop("test")

with open(yaml_path, "w") as f:
    yaml.safe_dump(fixed_yaml, f, sort_keys=False)

print(open(yaml_path).read())

path: /content/bowling_dataset
train: train/images
val: valid/images
test: test/images
names:
- ball
- car
- fallen-pins
- standing-pins



In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=416,
    batch=16,
    patience=10,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,

    degrees=10,
    translate=0.08,
    scale=0.3,
    shear=1,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.7,
    mixup=0.0,

    val=True,
    plots=True,
    save=True,

    project="/content/runs",
    name="bowling_fast_yolov8n"
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/bowling_dataset/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=bowling_fast_yolov8n-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m

In [ ]:
best_model = YOLO("/content/runs/bowling_fast_yolov8n-3/weights/best.pt")

metrics = best_model.val(
    data=str(yaml_path),
    imgsz=640,
    conf=0.25,
    iou=0.5
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 877.9±394.6 MB/s, size: 19.9 KB)
val: Scanning /content/bowling_dataset/valid/labels.cache... 220 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 220/220 102.5Mit/s 0.0s
val: /content/bowling_dataset/valid/images/VID_20260504_135512_idx00021_jpg.rf.2d25a2bd6b50e4a969c92b83d7529dae.jpg: 5 duplicate labels removed
val: /content/bowling_dataset/valid/images/VID_20260504_135512_idx00279_jpg.rf.e191de6074713347742d81ca679c9fe7.jpg: 8 duplicate labels removed
val: /content/bowling_dataset/valid/images/VID_20260504_135512_idx00591_jpg.rf.2703aceb1a9b1fcabd48e55d8fd471ec.jpg: 8 duplicate labels removed
val: /content/bowling_dataset/valid/images/VID_20260504_135512_idx00678_jpg.rf.8bf36dfbc9375ce35e79c189c98e6651.jpg: 8 duplicate labels removed
val: /content/bowling_dataset/valid/i

In [ ]:
from google.colab import files

files.download("/content/runs/bowling_final_yolov8s/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>